# Lab: from text to BERT



A sentence such as *"The cat sat on the mat."* goes through three stages before BERT can predict a hidden word in it: a **tokenizer** splits it into subword units, **self-attention** mixes information between those units while ignoring padding, and a **masked-language-model (MLM)** head scores every vocabulary entry for each hidden position. In this lab you build or test each stage yourself, then use a pretrained BERT to see the stages working together.

**Learning objectives**

- Apply learned BPE merges and WordPiece longest-match rules to new words, and explain how each handles unknown characters.
- Measure how vocabulary size changes sequence length, and how one tokenizer can be much less efficient for some languages than for others.
- Implement scaled dot-product attention with a padding mask and prove, with a test, that padded positions cannot change the result.
- Read BERT's three input tensors and implement BERT's 80/10/10 masking rule.
- Query a pretrained BERT, measure how well it recovers masked tokens, and reason carefully about what a template-based bias probe can and cannot show.

**Prerequisites:** [Attention Is All You Need](../../transformers/Attention_Is_All_You_Need.ipynb), the [BPE](../../tokenizers/BPE_tokenizer.ipynb) and [WordPiece](../../tokenizers/wordpiece_tokenizer.ipynb) notebooks, and [BERT](../../transformers/BERT.ipynb).

**Environment and compute**

- Written for a **Kaggle notebook with a GPU and internet access**. In Kaggle: *Settings → Accelerator → GPU*, and *Settings → Internet → on*. Kaggle images already include `torch`, `transformers`, `tokenizers`, `datasets`, and `matplotlib`.
- Parts A and B are light and would also run on a CPU. Part C runs BERT-base, which is much faster on the GPU.
- **Downloads** (cached by Hugging Face outside your working directory; nothing is written to `/kaggle/working`):
  - [WikiText-2 (raw)](https://huggingface.co/datasets/Salesforce/wikitext), a few MB of Wikipedia text, licensed CC BY-SA.
  - The tokenizers of `google-bert/bert-base-multilingual-cased` and `openai-community/gpt2` (a few MB each).
  - [`google-bert/bert-base-uncased`](https://huggingface.co/google-bert/bert-base-uncased) tokenizer and weights, about 440 MB (Apache-2.0).
- Plan about **two hours**. Seeds are fixed, but seeds do not guarantee identical numbers across hardware or library versions, so expect small differences from the trends described.

**How to work:** run the cells in order. Cells marked **TODO** contain a stub; until you complete them, `Run All` will stop at the first unfinished exercise (it raises `NotImplementedError`). Each exercise is followed by a check cell: if it prints "checks passed", your code meets the requirements tested (passing is necessary, not proof of full marks).

**Deliverable:** this notebook, run top to bottom, with all checks passing and all written answers filled in.

**Marks (30)**

| Part | Exercise | Marks |
| --- | --- | --- |
| A. Tokenization | 1 BPE tokenizer · 2 WordPiece tokenizer · 3 vocabulary-size study · 4 tokenizers across languages | 2 + 2 + 2 + 2 |
| B. Attention | 5 masked attention · 6 padding mask · 7 leakage test | 3 + 1 + 2 |
| C. BERT | 8 input tensors · 9 MLM masking · 10 top-k predictions · 11 bias probe | 2 + 4 + 3 + 2 |
| All | Written answers: correct, specific, and supported by your outputs | 5 |

## Setup

In [1]:
# Optional, only if a package is missing in your environment (Kaggle images already include these):
# !pip install -q transformers tokenizers datasets

import collections
import math
import random

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import datasets
import tokenizers
import transformers

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| transformers", transformers.__version__,
      "| tokenizers", tokenizers.__version__, "| datasets", datasets.__version__)
print("device:", device, "(" + torch.cuda.get_device_name(0) + ")" if device.type == "cuda" else "")
if device.type != "cuda":
    print("No GPU found: Parts A and B are fine, Part C will be slow. On Kaggle, enable the GPU accelerator.")

# Part A — Subword tokenization

Both BPE and WordPiece **learn** a vocabulary from a corpus and then **apply** it to new text. Training was covered in the notebooks; here you write the *application* step, which is what runs every time a model reads text.

| | BPE (GPT-2, RoBERTa) | WordPiece (BERT) |
| --- | --- | --- |
| What training produces | an ordered list of **merge rules** | a **vocabulary** of pieces (continuations start with `##`) |
| How a new word is split | start from characters, apply the merges **in the order they were learned** | **greedy longest match**: take the longest piece in the vocabulary that starts here, repeat |
| Unknown character | becomes `[UNK]`; the rest of the word is still split | the **whole word** becomes `[UNK]` |

## Exercise 1 — Apply BPE merges (2 marks)

Write `bpe_tokenize_word(word, merges, base_vocab)`.

- Start from the list of characters of `word`, replacing any character not in `base_vocab` with `"[UNK]"`.
- Go through `merges` **in order**. Each entry is `((a, b), merged)`. Replace every adjacent pair `a, b` in the current list by `merged`, scanning left to right, and move on to the next rule.

The toy rules below are the first three merges learned on the corpus `hug ×10, pug ×5, pun ×12, bun ×4, hugs ×5` from the BPE notebook.

In [2]:
TOY_BASE_VOCAB = set("bghnpsu")
TOY_MERGES = [(("u", "g"), "ug"), (("u", "n"), "un"), (("h", "ug"), "hug")]

In [3]:
def bpe_tokenize_word(word, merges, base_vocab, unk_token="[UNK]"):
    """Split one word by applying BPE merges in their learned order. Returns a list of tokens."""
    # TODO 1: start from characters; replace characters not in base_vocab with unk_token.
    # TODO 2: for each ((a, b), merged) in merges, rebuild the list, replacing each adjacent a, b with merged.
    raise NotImplementedError

In [4]:
cases = {
    "hugs":  ["hug", "s"],
    "bug":   ["b", "ug"],
    "pun":   ["p", "un"],
    "mug":   ["[UNK]", "ug"],
    "thug":  ["[UNK]", "hug"],
    "unhug": ["un", "hug"],
}
for word, expected in cases.items():
    got = bpe_tokenize_word(word, TOY_MERGES, TOY_BASE_VOCAB)
    assert got == expected, f"{word!r}: expected {expected}, got {got}"
    print(f"{word:6s} -> {got}")
print("Exercise 1 checks passed.")

**Question 1.** Suppose the merge list also contained `(("h", "u"), "hu")`, placed **before** `(("u", "g"), "ug")`. How would `hug` be tokenized now? What does this tell you about why merges must be applied in their learned order rather than, say, longest-first?

**Your answer — Question 1:** *(write here)*

## Exercise 2 — WordPiece longest match (2 marks)

Write `wordpiece_tokenize_word(word, vocab)`.

- Start at position `start = 0`. Try the **longest** substring `word[start:end]` first and shorten it one character at a time until it is in `vocab`. Pieces that do not begin the word must be looked up **with** the `##` prefix.
- Append the match and continue from where it ended.
- If no piece matches at some position, return `["[UNK]"]` for the **whole word**.

The toy vocabulary below is hand-made so that the expected answers are easy to verify by hand.

In [ ]:
TOY_WP_VOCAB = {"[UNK]", "b", "h", "p", "u", "##u", "##g", "##n", "##s", "hu", "hug", "##gs", "##un"}

In [ ]:
def wordpiece_tokenize_word(word, vocab, unk_token="[UNK]", prefix="##"):
    """Split one word by greedy longest match. Returns a list of tokens, or [unk_token] for the whole word."""
    # TODO: implement the loop described above.
    raise NotImplementedError

In [ ]:
cases = {
    "hugs":  ["hug", "##s"],          # longest match takes "hug", not "hu" + "##gs"
    "bun":   ["b", "##un"],
    "pugs":  ["p", "##u", "##gs"],
    "hu":    ["hu"],
    "mug":   ["[UNK]"],               # no piece starts with "m": the whole word is unknown
    "hugsx": ["[UNK]"],               # "x" fails at the end, and the whole word still becomes [UNK]
}
for word, expected in cases.items():
    got = wordpiece_tokenize_word(word, TOY_WP_VOCAB)
    assert got == expected, f"{word!r}: expected {expected}, got {got}"
    print(f"{word:6s} -> {got}")
print("Exercise 2 checks passed.")

**Question 2.** BPE turned `mug` into `['[UNK]', 'ug']`, while WordPiece turned it into `['[UNK]']`. Give one advantage of each behavior. Why does GPT-2's *byte-level* BPE never need an `[UNK]` token at all?

**Your answer — Question 2:** *(write here)*

## Exercise 3 — How vocabulary size changes sequence length (2 marks)

Real tokenizers are trained on large corpora, so we now use the Hugging Face `tokenizers` library, a fast (Rust) implementation, on **WikiText-2**. The cell below downloads it and removes empty lines and section headings.

Each trained tokenizer first **pre-tokenizes** text into words (splitting on whitespace and punctuation), then splits each word into subwords. Two numbers summarize how efficient a tokenizer is on some text:

- **Fertility** = subword tokens per pre-tokenized word. A value of 1.0 means no word was split; larger values mean longer sequences.
- **Split rate** = fraction of words split into more than one token.

> **Note on the library's WordPiece.** In `tokenizers`, `WordPieceTrainer` learns its vocabulary with BPE-style frequency merges rather than the likelihood score from the WordPiece notebook. What remains WordPiece-specific here is the `##` convention and greedy longest-match tokenization. So differences between the two below come mainly from **how the vocabulary is applied**, not from the training score.

In [ ]:
from datasets import load_dataset

wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

def clean(lines):
    """Drop empty lines and '= Heading =' lines."""
    return [s.strip() for s in lines if s.strip() and not s.strip().startswith("=")]

train_texts, val_texts = clean(wiki["train"]["text"]), clean(wiki["validation"]["text"])
print(f"train paragraphs: {len(train_texts):,} | validation paragraphs: {len(val_texts):,}")
print("example:", val_texts[0][:200], "...")

In [ ]:
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers

def train_subword_tokenizer(kind, texts, vocab_size):
    """Train a lowercasing BPE or WordPiece tokenizer on `texts` (training split only)."""
    if kind == "bpe":
        tok = Tokenizer(models.BPE(unk_token="[UNK]"))
        trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["[UNK]"])
    elif kind == "wordpiece":
        tok = Tokenizer(models.WordPiece(unk_token="[UNK]"))
        trainer = trainers.WordPieceTrainer(vocab_size=vocab_size, special_tokens=["[UNK]"])
    else:
        raise ValueError(kind)
    tok.normalizer = normalizers.Lowercase()
    tok.pre_tokenizer = pre_tokenizers.Whitespace()      # words = runs of letters/digits, or of punctuation
    tok.train_from_iterator(texts, trainer=trainer)
    return tok

demo = train_subword_tokenizer("wordpiece", train_texts, vocab_size=4000)
enc = demo.encode("Tokenization is unbelievably useful.")
print("tokens  :", enc.tokens)
print("word_ids:", enc.word_ids, " <- index of the pre-tokenized word each token came from")

Write `subword_stats(tokenizer, texts)`. Use `tokenizer.encode_batch(texts)`, and for each encoding use `enc.word_ids` (as printed above) to count how many tokens each word produced. Return a dictionary with `fertility`, `split_rate`, and `unk_rate` (fraction of all tokens equal to `"[UNK]"`).

In [ ]:
def subword_stats(tokenizer, texts):
    """Return {"fertility": tokens per word, "split_rate": fraction of words with >1 token, "unk_rate": [UNK] share}."""
    n_words = n_tokens = n_split = n_unk = 0
    for enc in tokenizer.encode_batch(texts):
        # TODO: count tokens per word using enc.word_ids (ignore entries that are None),
        #       and count [UNK] tokens using enc.tokens.
        raise NotImplementedError
    return {"fertility": n_tokens / n_words, "split_rate": n_split / n_words, "unk_rate": n_unk / n_tokens}

In [ ]:
toy = subword_stats(demo, ["tokenization is useful"])
assert set(toy) == {"fertility", "split_rate", "unk_rate"}
assert toy["fertility"] >= 1.0 and 0.0 <= toy["split_rate"] <= 1.0
manual = demo.encode("tokenization is useful")
assert abs(toy["fertility"] - len(manual.tokens) / 3) < 1e-9, "fertility should be tokens / words (3 words here)"
print("Exercise 3 checks passed. Now running the vocabulary-size study (a minute or so)...")

VOCAB_SIZES = [1000, 4000, 16000]
study = {}
for kind in ("bpe", "wordpiece"):
    for size in VOCAB_SIZES:
        tok = train_subword_tokenizer(kind, train_texts, size)              # trained on train only
        study[(kind, size)] = (tok, subword_stats(tok, val_texts))          # measured on validation
        s = study[(kind, size)][1]
        print(f"{kind:9s} V={size:6,d}  fertility={s['fertility']:.3f}  split_rate={s['split_rate']:.3f}  unk_rate={s['unk_rate']:.4f}")

fig, ax = plt.subplots(figsize=(5.5, 3.4))
for kind, marker in (("bpe", "o"), ("wordpiece", "s")):
    ax.plot(VOCAB_SIZES, [study[(kind, v)][1]["fertility"] for v in VOCAB_SIZES], marker=marker, label=kind)
ax.set_xscale("log"); ax.set_xticks(VOCAB_SIZES); ax.set_xticklabels([f"{v:,}" for v in VOCAB_SIZES])
ax.set_xlabel("vocabulary size (log scale)"); ax.set_ylabel("tokens per word (validation)")
ax.set_title("Larger vocabularies give shorter sequences"); ax.legend(); plt.tight_layout(); plt.show()

for word in ["photosynthesis", "unbelievably", "bennett", "transformers", "kaggle"]:
    print(f"{word:15s}", " | ".join(f"{k[0]} {v:>5}: {study[(k, v)][0].encode(word).tokens}"
                                    for k in ("bpe", "wordpiece") for v in (1000, 16000)))

**Question 3.** (a) Describe the trend in fertility as the vocabulary grows, using your numbers. (b) The vocabulary also sets the size of the embedding table. For a model with hidden size $d = 768$, how many embedding parameters does a 16,000-token vocabulary need, compared with 1,000 tokens? (c) Using (a) and (b), explain the trade-off a model designer faces when choosing the vocabulary size. Why are all the measurements taken on the *validation* paragraphs?

**Your answer — Question 3:** *(write here)*

## Exercise 4 — One sentence, three pretrained tokenizers, three languages (2 marks)

A tokenizer trained mostly on one language can be very inefficient on others. We compare three real tokenizers on the same short sentence written in English, Hindi (Devanagari script), and Hinglish (Hindi written in the Latin alphabet, as in many chats).

| Tokenizer | Algorithm | Trained on |
| --- | --- | --- |
| `google-bert/bert-base-uncased` | WordPiece, lowercases and strips accents | English |
| `google-bert/bert-base-multilingual-cased` | WordPiece | Wikipedia in ~100 languages |
| `openai-community/gpt2` | byte-level BPE | English web text |

Write `tokens_per_word(tokenizer, text)`: the number of tokens from `tokenizer.tokenize(text)` (which excludes special tokens such as `[CLS]`) divided by the number of whitespace-separated words, `len(text.split())`.

In [ ]:
from transformers import AutoTokenizer

PRETRAINED = ["google-bert/bert-base-uncased", "google-bert/bert-base-multilingual-cased", "openai-community/gpt2"]
pretrained_toks = {name: AutoTokenizer.from_pretrained(name) for name in PRETRAINED}   # tokenizers only: small downloads

SENTENCES = {
    "English":  "I like Indian food very much.",
    "Hindi":    "मुझे भारतीय खाना बहुत पसंद है।",
    "Hinglish": "Mujhe Indian khana bahut pasand hai.",
}

In [ ]:
def tokens_per_word(tokenizer, text):
    """Tokens produced by tokenizer.tokenize(text), divided by the number of whitespace-separated words."""
    # TODO
    raise NotImplementedError

In [ ]:
assert abs(tokens_per_word(pretrained_toks[PRETRAINED[0]], "hello world") - len(pretrained_toks[PRETRAINED[0]].tokenize("hello world")) / 2) < 1e-9
print("Exercise 4 checks passed.\n")

print(f"{'':10s}" + "".join(f"{name.split('/')[-1]:>30s}" for name in PRETRAINED))
for lang, text in SENTENCES.items():
    print(f"{lang:10s}" + "".join(f"{tokens_per_word(pretrained_toks[n], text):30.2f}" for n in PRETRAINED))

for lang, text in SENTENCES.items():
    print(f"\n{lang}: {text}")
    for name in PRETRAINED:
        tok = pretrained_toks[name]
        pieces = tok.tokenize(text)
        print(f"  {name.split('/')[-1]:28s} {len(pieces):2d} tokens: {pieces}")
        if "uncased" in name:
            print(f"  {'':28s} decoded back: {tok.decode(tok.encode(text, add_special_tokens=False))!r}")

GPT-2's tokens for Hindi look like `à¤...` because byte-level BPE works on UTF-8 **bytes**: each Devanagari character takes 3 bytes, and the tokenizer shows each byte as a printable stand-in character.

**Question 4.** (a) Which tokenizer is most efficient for each language, and which is least efficient for Hindi? Explain using the table above. (b) Compare the decoded `bert-base-uncased` output for Hindi with the original sentence. What has happened? (c) Context windows are measured in tokens, and paid APIs often charge per token. What does your table imply for users who write in Hindi?

**Your answer — Question 4:** *(write here)*

# Part B — Self-attention with a padding mask

Sentences in a batch have different lengths, so shorter ones are **padded** to a common length. Self-attention lets every position read every other position, so without a mask the real tokens would also read the padding, and their representations would depend on how much padding the batch happened to need.

We use the same convention as [Attention Is All You Need](../../transformers/Attention_Is_All_You_Need.ipynb): a Boolean mask in which **`True` means "may attend"** and `False` means "blocked". (`F.scaled_dot_product_attention` uses the same convention. Be careful: the [GPT-2 lab](../gpt2_causal_lm/gpt2_causal_lm_lab.ipynb) uses the opposite one, `True` = blocked. Always check which convention a piece of code uses.)

The tensors in this part are tiny, so they stay on the CPU.

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V,\qquad M_{ij}=\begin{cases}0&\text{key } j \text{ allowed}\\-\infty&\text{key } j \text{ blocked}\end{cases}$$

## Exercise 5 — Scaled dot-product attention with a mask (3 marks)

Write `scaled_dot_product_attention(q, k, v, mask=None)`.

- `q`: `(B, H, T_q, d_k)`, `k` and `v`: `(B, H, T_k, d_k)`. `mask`: Boolean, broadcastable to `(B, H, T_q, T_k)`, `True` = may attend.
- Compute the scores, scale by $\sqrt{d_k}$, set blocked scores to `-inf` (use `masked_fill`), apply softmax over the **key** dimension, and combine the values.
- Return `(output, weights)` with shapes `(B, H, T_q, d_k)` and `(B, H, T_q, T_k)`.

In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    """Returns (output, weights). mask: bool, broadcastable to (B, H, T_q, T_k); True = may attend."""
    # TODO 1: scores = q k^T / sqrt(d_k), shape (B, H, T_q, T_k)
    # TODO 2: if a mask is given, fill blocked positions (mask == False) with -inf
    # TODO 3: softmax over the last dimension, then multiply by v
    raise NotImplementedError

In [ ]:
g = torch.Generator().manual_seed(0)
B, H, T, d_k = 2, 3, 5, 4
q, k, v = (torch.randn(B, H, T, d_k, generator=g) for _ in range(3))
lengths = torch.tensor([5, 3])                                            # sequence 2 has 2 padded positions
key_mask = (torch.arange(T)[None, :] < lengths[:, None])[:, None, None, :]   # (B, 1, 1, T), True = real token

out, w = scaled_dot_product_attention(q, k, v, key_mask)
assert out.shape == (B, H, T, d_k) and w.shape == (B, H, T, T), "wrong output shapes"
assert torch.allclose(w.sum(-1), torch.ones(B, H, T)), "each row of weights must sum to 1"
assert (w[1, :, :, 3:] == 0).all(), "padded keys must receive exactly zero weight"
ref = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask)
assert torch.allclose(out, ref, atol=1e-5), "does not match torch's scaled_dot_product_attention"
out_nomask, _ = scaled_dot_product_attention(q, k, v)
assert torch.allclose(out_nomask, F.scaled_dot_product_attention(q, k, v), atol=1e-5), "mask=None case is wrong"
print("Exercise 5 checks passed.")

## Exercise 6 — The padding mask (1 mark)

Write `make_padding_mask(input_ids, pad_id)`: given token IDs of shape `(B, T)`, return a Boolean mask of shape `(B, 1, 1, T)` that is `True` at real tokens. The two size-1 dimensions broadcast over heads and over **query** positions: every query may read every real **key**.

In [ ]:
def make_padding_mask(input_ids, pad_id):
    """(B, T) token IDs -> (B, 1, 1, T) bool mask, True at non-padding positions."""
    # TODO
    raise NotImplementedError

In [ ]:
ids = torch.tensor([[5, 6, 7, 0, 0], [8, 9, 10, 11, 12]])
m = make_padding_mask(ids, pad_id=0)
assert m.shape == (2, 1, 1, 5) and m.dtype == torch.bool
assert m[0, 0, 0].tolist() == [True, True, True, False, False] and m[1].all()
print("Exercise 6 checks passed.")

## Provided: a small Transformer encoder

Read this code; you do not need to change it. It is the encoder from *Attention Is All You Need* in compact form (post-norm, sinusoidal positions), and it calls **your** `scaled_dot_product_attention`. Shapes: IDs `(B, T)` → hidden states `(B, T, D)`.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.W_q, self.W_k, self.W_v, self.W_o = (nn.Linear(d_model, d_model) for _ in range(4))

    def forward(self, x, mask=None):                                   # x: (B, T, D)
        B, T, D = x.shape
        split = lambda t: t.view(B, T, self.h, self.d_k).transpose(1, 2)        # (B, H, T, d_k)
        out, weights = scaled_dot_product_attention(split(self.W_q(x)), split(self.W_k(x)), split(self.W_v(x)), mask)
        return self.W_o(out.transpose(1, 2).reshape(B, T, D)), weights          # merge heads -> (B, T, D)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        a, _ = self.attn(x, mask)
        x = self.norm1(x + self.drop(a))                  # residual + layer norm (post-norm, as in the paper)
        return self.norm2(x + self.drop(self.ff(x)))


class TinyEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=32, n_heads=4, d_ff=64, n_layers=2, max_len=64, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        pos = torch.arange(max_len)[:, None]
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe)
        self.layers = nn.ModuleList(EncoderLayer(d_model, n_heads, d_ff) for _ in range(n_layers))

    def forward(self, ids, mask=None):                   # ids: (B, T)
        x = self.emb(ids) * math.sqrt(self.emb.embedding_dim) + self.pe[: ids.size(1)]
        for layer in self.layers:
            x = layer(x, mask)
        return x                                          # (B, T, D)

torch.manual_seed(SEED)
encoder = TinyEncoder(vocab_size=50).eval()               # eval(): dropout off, so outputs are deterministic
print(encoder(torch.tensor([[3, 4, 5, 0]]), make_padding_mask(torch.tensor([[3, 4, 5, 0]]), 0)).shape)

## Exercise 7 — Prove that padding cannot leak (2 marks)

A mask is only trustworthy if we test it. Write `padding_leakage(model, ids, junk_ids, pad_id, use_mask)`:

- `ids` is a padded batch. `junk_ids` is identical **except** that the padded positions hold other, arbitrary token IDs.
- Build **one** mask from `ids` and pass that same mask to both forward passes (or pass `None` for both when `use_mask=False`). This simulates "the content of a masked position changed, nothing else did".
- Return the largest absolute difference between the two outputs, taken **only over real (non-padding) positions**.

With the mask, the difference should be zero (up to floating-point error); without it, it should not be.

In [ ]:
@torch.no_grad()
def padding_leakage(model, ids, junk_ids, pad_id, use_mask=True):
    """Max |output difference| at real positions when only padded positions change."""
    # TODO 1: mask = make_padding_mask(ids, pad_id) if use_mask else None
    # TODO 2: run the model on ids and on junk_ids with the SAME mask
    # TODO 3: select real positions (ids != pad_id) and return the max absolute difference as a float
    raise NotImplementedError

In [ ]:
ids = torch.tensor([[7, 8, 9, 0, 0, 0], [10, 11, 12, 13, 14, 0]])
junk_ids = ids.clone()
junk_ids[ids == 0] = torch.tensor([31, 32, 33, 34])       # change only the padded positions
with_mask = padding_leakage(encoder, ids, junk_ids, pad_id=0, use_mask=True)
without_mask = padding_leakage(encoder, ids, junk_ids, pad_id=0, use_mask=False)
print(f"with mask   : max change at real positions = {with_mask:.2e}")
print(f"without mask: max change at real positions = {without_mask:.2e}")
assert with_mask < 1e-5, "with the mask, real positions must not depend on padded content"
assert without_mask > 1e-3, "without the mask, padded content should leak into real positions"
print("Exercise 7 checks passed.")

**Question 5 (shapes and masks).** (a) For the encoder above with `B = 2`, `T = 6`, `D = 32`, `H = 4`, give the shapes of: the input IDs, the embeddings, `q` after splitting into heads, the attention weights, and the encoder output. (b) The padding mask hides padded **keys**, but the padded positions are still **queries** and still produce output vectors. Why is that harmless for BERT? (c) BERT uses a padding mask but no causal mask, while GPT-2 uses both. Explain why in one or two sentences each.

**Your answer — Question 5:** *(write here)*

# Part C — BERT: inputs, objective, and predictions

We now load the pretrained `bert-base-uncased` with its masked-language-model head (about 440 MB on the first run) and move it to the GPU.

In [ ]:
from transformers import AutoModelForMaskedLM

BERT = "google-bert/bert-base-uncased"
bert_tok = AutoTokenizer.from_pretrained(BERT)
bert_mlm = AutoModelForMaskedLM.from_pretrained(BERT).to(device).eval()     # eval(): dropout off
print("vocabulary size:", bert_tok.vocab_size, "| parameters:", round(sum(p.numel() for p in bert_mlm.parameters()) / 1e6, 1), "M")
print("special tokens:", {t: bert_tok.convert_tokens_to_ids(t) for t in ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]})

## Exercise 8 — BERT's three input tensors (2 marks)

For a **sentence pair**, BERT's tokenizer returns:

| Tensor | Meaning |
| --- | --- |
| `input_ids` | `[CLS] sentence A [SEP] sentence B [SEP] [PAD] ...` as vocabulary IDs |
| `token_type_ids` | segment of each token: 0 for sentence A, 1 for sentence B |
| `attention_mask` | 1 for real tokens, 0 for padding: the padding mask of Part B in the form Hugging Face uses |

Write `describe_bert_input(enc, tokenizer)` for a single encoded pair (batch size 1). Return a dictionary with:
`cls_position` (index of `[CLS]`), `sep_positions` (list of indices of `[SEP]`), `segment_b_tokens` (number of **real** tokens with `token_type_ids == 1`), and `pad_tokens` (number of padding positions).

In [ ]:
pair = bert_tok("The cat sat on the mat.", "It looked very comfortable.",
                padding="max_length", max_length=20, return_tensors="pt")
print(f"{'pos':>3}  {'token':12s} {'input_id':>8} {'type':>4} {'attn':>4}")
for i, (tid, tt, am) in enumerate(zip(pair["input_ids"][0], pair["token_type_ids"][0], pair["attention_mask"][0])):
    print(f"{i:3d}  {bert_tok.convert_ids_to_tokens(int(tid)):12s} {int(tid):8d} {int(tt):4d} {int(am):4d}")

In [ ]:
def describe_bert_input(enc, tokenizer):
    """Summarize one encoded sentence pair (batch size 1)."""
    ids, types, attn = enc["input_ids"][0], enc["token_type_ids"][0], enc["attention_mask"][0]
    # TODO: compute and return the four values described above.
    raise NotImplementedError

In [ ]:
info = describe_bert_input(pair, bert_tok)
print(info)
tokens_b = bert_tok.tokenize("It looked very comfortable.")
assert info["cls_position"] == 0
assert len(info["sep_positions"]) == 2
assert info["segment_b_tokens"] == len(tokens_b) + 1, "segment B = sentence-B tokens plus its closing [SEP]"
assert info["pad_tokens"] == 20 - (info["sep_positions"][-1] + 1)
print("Exercise 8 checks passed.")

**Question 6.** Which segment does the *first* `[SEP]` belong to, and which the second? What `token_type_ids` do padding positions get, and why does it not matter? What would `[CLS]`'s final hidden state be used for when BERT is later adapted to a sentence-pair classification task?

**Your answer — Question 6:** *(write here)*

## Exercise 9 — BERT's 80/10/10 masking rule (4 marks)

During pretraining, BERT selects **15%** of the eligible tokens as prediction targets. Of the selected tokens:

- **80%** are replaced by `[MASK]`,
- **10%** are replaced by a **random** token from the vocabulary,
- **10%** are left **unchanged**.

The model must predict the original token at every selected position. Non-selected positions get the label `-100`, which PyTorch's cross-entropy ignores. Special tokens (`[CLS]`, `[SEP]`) and padding are never selected.

Write `mask_tokens(input_ids, special_mask, mask_token_id, vocab_size, mlm_prob, generator)`. It must **not** modify `input_ids` in place. Return `(masked_ids, labels)`.

*Hint:* draw the random decisions with `torch.bernoulli(probabilities, generator=generator)` and the random tokens with `torch.randint(vocab_size, shape, generator=generator)`. For the 10%/10% split, decide among the selected-but-not-`[MASK]` tokens with probability 0.5.

In [ ]:
# A batch of real text: 512 validation paragraphs, truncated to 128 tokens.
mlm_batch = bert_tok(val_texts[:512], padding=True, truncation=True, max_length=128,
                     return_tensors="pt", return_special_tokens_mask=True)
special_mask = mlm_batch["special_tokens_mask"].bool() | (mlm_batch["attention_mask"] == 0)   # never select these
print("batch shape:", tuple(mlm_batch["input_ids"].shape), "| eligible tokens:", int((~special_mask).sum()))

In [ ]:
def mask_tokens(input_ids, special_mask, mask_token_id, vocab_size, mlm_prob=0.15, generator=None):
    """BERT MLM masking. Returns (masked_ids, labels); labels are -100 where no prediction is required."""
    labels = input_ids.clone()
    masked_ids = input_ids.clone()
    # TODO 1: select each eligible token with probability mlm_prob (special/pad positions: probability 0)
    # TODO 2: labels = -100 wherever a token was not selected
    # TODO 3: 80% of selected -> mask_token_id
    # TODO 4: half of the remaining selected tokens -> a random token ID; the rest stay unchanged
    raise NotImplementedError
    return masked_ids, labels

In [ ]:
original = mlm_batch["input_ids"].clone()
masked_ids, labels = mask_tokens(mlm_batch["input_ids"], special_mask, bert_tok.mask_token_id, bert_tok.vocab_size,
                                 mlm_prob=0.15, generator=torch.Generator().manual_seed(SEED))

assert torch.equal(mlm_batch["input_ids"], original), "input_ids was modified in place"
selected = labels != -100
assert not (selected & special_mask).any(), "special tokens or padding were selected"
assert torch.equal(labels[selected], original[selected]), "labels must hold the ORIGINAL token at selected positions"
assert torch.equal(masked_ids[~selected], original[~selected]), "non-selected positions must be unchanged"

eligible = int((~special_mask).sum()); n_sel = int(selected.sum())
is_mask = (masked_ids == bert_tok.mask_token_id) & selected
is_unchanged = (masked_ids == original) & selected
is_random = selected & ~is_mask & ~is_unchanged
frac = lambda m: int(m.sum()) / n_sel
print(f"selected: {n_sel / eligible:.3f} of eligible tokens (target 0.15)")
print(f"of selected -> [MASK]: {frac(is_mask):.3f} (0.8) | random: {frac(is_random):.3f} (~0.1) | unchanged: {frac(is_unchanged):.3f} (~0.1)")
assert abs(n_sel / eligible - 0.15) < 0.01
assert abs(frac(is_mask) - 0.8) < 0.03 and abs(frac(is_random) - 0.1) < 0.03 and abs(frac(is_unchanged) - 0.1) < 0.03
print("Exercise 9 checks passed.")

**Question 7.** (a) Why not simply replace all selected tokens by `[MASK]`? Think about what the model sees when it is later used on normal text. (b) Why keep 10% of selected tokens unchanged, and why replace 10% with random tokens? (c) A random replacement occasionally picks the original token, so the "random" share you measured can be slightly below 0.1. Why is that harmless?

**Your answer — Question 7:** *(write here)*

## Exercise 10 — BERT's top-k predictions (3 marks)

Write `top_k_predictions(text, k)`. `text` contains exactly one `[MASK]`. Tokenize it, move the tensors to `device`, run `bert_mlm`, find the `[MASK]` position, apply softmax to its logits over the vocabulary, and return a list of `k` pairs `(token_string, probability)`, most probable first.

In [ ]:
@torch.no_grad()
def top_k_predictions(text, k=5):
    """[(token, probability), ...] for the single [MASK] in text, most probable first."""
    enc = bert_tok(text, return_tensors="pt").to(device)
    # TODO 1: logits = bert_mlm(**enc).logits, shape (1, T, V)
    # TODO 2: find the index of bert_tok.mask_token_id in enc["input_ids"][0]
    # TODO 3: softmax over the vocabulary at that index, take topk, convert IDs to token strings
    raise NotImplementedError

In [ ]:
from transformers import pipeline

preds = top_k_predictions("The cat sat on the [MASK].", k=5)
print(preds)
probs_ = [p for _, p in preds]
assert len(preds) == 5 and probs_ == sorted(probs_, reverse=True) and 0 < sum(probs_) <= 1 + 1e-6
fill = pipeline("fill-mask", model=bert_mlm, tokenizer=bert_tok, device=0 if device.type == "cuda" else -1)
ref = fill("The cat sat on the [MASK].", top_k=1)[0]
assert bert_tok.convert_tokens_to_ids(preds[0][0]) == ref["token"], "top-1 token differs from the fill-mask pipeline"
assert abs(preds[0][1] - ref["score"]) < 1e-3, "top-1 probability differs from the fill-mask pipeline"
print("Exercise 10 checks passed.\n")

for text in ["Paris is the capital of [MASK].",
             "She opened the [MASK] and walked into the room.",
             "The doctor gave me some [MASK] for my headache.",
             "Bennett University is located in [MASK]."]:
    print(f"{text:50s}", [(t, round(p, 3)) for t, p in top_k_predictions(text)])

### Provided: how often does BERT recover masked tokens?

Your `mask_tokens` output from Exercise 9 is a realistic MLM test set. The cell below runs BERT on it and measures, **separately** for the three kinds of selected positions, how often BERT's most probable token is the original. It also reports the MLM loss that pretraining minimizes (cross-entropy over selected positions only; `labels = -100` elsewhere). This is evaluation only: no weights change. Note that WikiText comes from Wikipedia, which was also part of BERT's pretraining data, so these numbers are likely optimistic.

In [ ]:
@torch.no_grad()
def mlm_eval(masked_ids, labels, attention_mask, batch_size=64):
    correct = {"[MASK]": 0, "random": 0, "unchanged": 0}; total = dict.fromkeys(correct, 0); losses = []
    for s in range(0, masked_ids.size(0), batch_size):
        ids_b, lab_b, att_b = (t[s:s + batch_size].to(device) for t in (masked_ids, labels, attention_mask))
        out = bert_mlm(input_ids=ids_b, attention_mask=att_b, labels=lab_b)    # HF computes CE, ignoring -100
        losses.append(out.loss.item())
        pred = out.logits.argmax(-1)
        sel = lab_b != -100
        kinds = {"[MASK]": sel & (ids_b == bert_tok.mask_token_id),
                 "unchanged": sel & (ids_b == lab_b)}
        kinds["random"] = sel & ~kinds["[MASK]"] & ~kinds["unchanged"]
        for name, m in kinds.items():
            correct[name] += int((pred[m] == lab_b[m]).sum()); total[name] += int(m.sum())
    for name in correct:
        print(f"{name:10s} positions: {total[name]:6d}   top-1 accuracy = {correct[name] / max(total[name], 1):.3f}")
    print(f"mean MLM loss over batches = {sum(losses) / len(losses):.3f} nats/token")

mlm_eval(masked_ids, labels, mlm_batch["attention_mask"])

**Question 8.** Rank the three kinds of positions by accuracy, and explain the ranking. Why would it be wrong to report the accuracy at `unchanged` positions as evidence of how well BERT 'understands' text?

**Your answer — Question 8:** *(write here)*

## Exercise 11 — A careful bias probe (2 marks)

Masked-word predictions are often used to look for social bias: for example, does BERT prefer `he` or `she` for a person described by an occupation? Write `pronoun_probs(sentence)`: for a sentence with one `[MASK]`, return `(P("he"), P("she"))`, the softmax probabilities of those two tokens at the mask position (over the **whole** vocabulary, not renormalized). Then run the provided table with **two different templates**.

In [ ]:
@torch.no_grad()
def pronoun_probs(sentence):
    """(P('he'), P('she')) at the single [MASK] in sentence, from the full softmax over the vocabulary."""
    # TODO: same steps as top_k_predictions, then read the probabilities of the token IDs for "he" and "she"
    raise NotImplementedError

In [ ]:
p_he, p_she = pronoun_probs("Yesterday [MASK] went to the market.")
assert 0 <= p_he <= 1 and 0 <= p_she <= 1 and p_he + p_she <= 1 + 1e-6
print("Exercise 11 checks passed.\n")

OCCUPATIONS = ["nurse", "receptionist", "teacher", "doctor", "engineer", "pilot", "programmer", "scientist"]
TEMPLATES = ["the {occ} said that [MASK] was tired.",
             "after the meeting, the {occ} explained why [MASK] had to leave early."]
fig, axes = plt.subplots(1, len(TEMPLATES), figsize=(11, 3.4), sharey=True)
for ax, template in zip(axes, TEMPLATES):
    log_ratios = []
    print(template)
    for occ in OCCUPATIONS:
        he, she = pronoun_probs(template.format(occ=occ))
        log_ratios.append(math.log(he / she))
        print(f"  {occ:13s} P(he)={he:.3f}  P(she)={she:.3f}  log(P(he)/P(she))={log_ratios[-1]:+.2f}")
    ax.barh(OCCUPATIONS, log_ratios, color=["#c0392b" if r < 0 else "#2b6cb0" for r in log_ratios])
    ax.axvline(0, color="#333333", lw=1); ax.set_xlabel("log P(he) / P(she)   (<0: 'she' preferred)")
    ax.set_title(template.replace("{occ}", "<occupation>"), fontsize=9)
plt.tight_layout(); plt.show()

**Question 9.** (a) Summarize what the two charts show. Are the occupation rankings the same for both templates? (b) Give at least **three** reasons why this experiment does *not* establish how biased BERT, or a system built on it, would be in practice. (c) Suggest one change to the experiment that would make its conclusions more reliable.

**Your answer — Question 9:** *(write here)*

## Wrap-up

You followed one sentence through every stage of BERT: subword tokenization (Part A), masked self-attention (Part B), and the masked-language-model objective and predictions (Part C). Before you submit, check:

- [ ] The notebook runs top to bottom in a fresh session, and every check prints "checks passed".
- [ ] All nine written answers are filled in and refer to **your** outputs, not only to general facts.
- [ ] You noted anything that surprised you, such as a tokenizer result or a prediction, in the relevant answer.